# Modeling

In [ ]:
import pandas as pd
from sklearn.decomposition import PCA
import numpy as np

from sklearn.linear_model import Ridge, Lasso, ElasticNet, BayesianRidge, HuberRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor,
    ExtraTreesRegressor
)

import xgboost as xgb
from sklearn.pipeline import Pipeline
from sklearn import model_selection
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV


from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.metrics import root_mean_squared_log_error
from numpy import expm1
from pathlib import Path


In [ ]:
# PATH DEFINITIONS
BASE_DIR = Path().resolve().parent

DATA_DIR = BASE_DIR / "data"
MODEL_DATA_DIR = DATA_DIR / "data_model"

In [ ]:
df_train = pd.read_csv(MODEL_DATA_DIR / "train_model.csv")

In [ ]:
X_train = df_train.drop(columns=['SALEPRICE'])
y_train = df_train['SALEPRICE']

In [ ]:
df_train.head()

In [ ]:
pca_analysis = PCA(random_state=0) 
pca_analysis.fit(X_train)

individual_variance = pca_analysis.explained_variance_ratio_
accumulated_variance = individual_variance.cumsum()

pca_table = pd.DataFrame({
    'COMPONENT': range(1, len(accumulated_variance) + 1),
    'INDIVIDUAL_VARIANCE': (individual_variance).round(4),
    'ACCUMULATED_VARIANCE': (accumulated_variance).round(4),
    'DELTA': (pd.Series(accumulated_variance).diff().fillna(accumulated_variance[0]) * 100).round(4)
})

first_above_95 = np.argmax(np.array(pca_table['ACCUMULATED_VARIANCE']) >= 0.95)
first_above_95

In [ ]:
models = {
    'Ridge':             Ridge(random_state=0),
    'Lasso':             Lasso(random_state=0),
    'ElasticNet':        ElasticNet(random_state=0),
    'Bayesian Ridge':    BayesianRidge(),
    'Huber':             HuberRegressor(max_iter=500),
    'KNN Regressor':     KNeighborsRegressor(),
    'SVR':               SVR(),
    'Random Forest':     RandomForestRegressor(random_state=0),
    'Extra Trees':       ExtraTreesRegressor(random_state=0),
    'Gradient Boosting': GradientBoostingRegressor(random_state=0),
    'XGBoost:': xgb.XGBRegressor(random_state=0)
}

kfold = model_selection.KFold(n_splits=5, shuffle=True, random_state=0)

In [ ]:
results = []
for name, model in models.items():
    pipeline = Pipeline([
        ('model', model)
    ])
    scores = model_selection.cross_validate(
        pipeline,
        X_train,
        y_train,
        scoring='neg_root_mean_squared_log_error',
        cv=kfold,
        n_jobs=-1
    )
    results.append({
        'MODEL': name.upper(),
        'RMSLE_MEAN': -scores['test_score'].mean(),
        'RMSLE_STD':   scores['test_score'].std()
    })

In [ ]:
df_exploration = pd.DataFrame(results)
df_exploration.columns = df_exploration.columns.str.upper()
df_exploration = df_exploration.round(4)

df_exploration = df_exploration.sort_values(by='RMSLE_MEAN').reset_index(drop=True)

df_exploration

## GRADIENT BOOSTING

In [ ]:
df_exploration_analysis = df_exploration.copy()
df_exploration_analysis[['RMSLE_MEAN', 'RMSLE_STD']] = df_exploration_analysis[['RMSLE_MEAN', 'RMSLE_STD']].rank(0, numeric_only=True, method='min', ascending=True)
df_exploration_analysis['POINTS'] = df_exploration_analysis['RMSLE_MEAN'] + df_exploration_analysis['RMSLE_STD'] 

df_exploration_analysis = df_exploration_analysis.sort_values(by='POINTS', ascending=True).reset_index(drop=True)
df_exploration_analysis

In [ ]:
gb_model = GradientBoostingRegressor(random_state=0)

param_distributions = {
        "n_estimators": [500, 525, 550],
        "learning_rate": [0.045, 0.05, 0.06],
        "max_depth": [6, 7], # experimentacao
        "min_samples_split": [90, 100, 110], #dobro do min_sample_leaf
        "min_samples_leaf": [45, 50, 55], # Geralmente, na faixa de 1-5%
        "max_features": [0.8, 0.85, 0.90], # um valor classico é \sqrt (no caso, 4 = 0.25)
        "subsample": [0.80, 0.85, 0.90] # depende do tamanho do modelo, cria generalização
    }

random_search = RandomizedSearchCV(
        estimator=gb_model,
        param_distributions=param_distributions,
        n_jobs=-1,
        random_state=0,
        scoring='neg_root_mean_squared_log_error'
    )

random_search.fit(X_train, y_train)

model_best_gb = random_search.best_estimator_
model_best_gb_score = random_search.best_score_

print(model_best_gb_score)
print(model_best_gb)

In [ ]:
param_grid_refined = {
    'learning_rate':     [0.05],
    'max_depth':         [4, 5, 6],     
    'max_features':      [0.7, 0.8, 1.0],
    'min_samples_leaf':  [45, 50, 55], 
    'min_samples_split': [140, 160, 180], 
    'n_estimators':      [225, 275, 325],
    'subsample':         [0.45, 0.55],
}

grid_search = GridSearchCV(
    estimator=gb_model,
    param_grid=param_grid_refined,
    n_jobs=-1,
    scoring='neg_root_mean_squared_log_error',
    return_train_score=True
)

grid_search.fit(X_train, y_train)

model_best_gb = grid_search.best_estimator_
model_best_gb_score = grid_search.best_score_
y_pred = model_best_gb.predict(X_train)

print(round(root_mean_squared_log_error(y_true=y_train, y_pred=y_pred), 4))
print(model_best_gb_score)
print(model_best_gb)

In [ ]:
# ANALISE HERE
gb_model_gap = grid_search.cv_results_['mean_train_score'][grid_search.best_index_] - grid_search.best_score_

print("GAP: ",gb_model_gap )

## RANDOM FOREST

In [ ]:
rf_model = RandomForestRegressor(random_state=0)

param_distributions = {
    "n_estimators": [400, 500, 600],
    "min_samples_leaf": [35,40,45],
    "max_features": ['sqrt', 0.8, 0.9],
    "max_samples": [0.7,0.8,0.9]
}

random_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=param_distributions,
    n_jobs=-1,
    random_state=0,
    scoring='neg_root_mean_squared_log_error',
    return_train_score=True
)

random_search.fit(X_train, y_train)

In [ ]:
rf_model = RandomForestRegressor(random_state=0)

param_grid = {
    "n_estimators": [400, 500, 600],
    "min_samples_leaf": [35, 40, 45],
    "max_depth": [4,5,6],
    "max_features": [0.7, 0.8, 0.9],
    "max_samples": [0.7, 0.8, 0.9]
}

grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    n_jobs=-1,
    scoring='neg_root_mean_squared_log_error',
    return_train_score=True
)

grid_search.fit(X_train, y_train)

In [ ]:
model_best_rf = grid_search.best_estimator_
model_best_rf_score = grid_search.best_score_
y_pred = model_best_rf.predict(X_train)
rf_model_gap = grid_search.cv_results_['mean_train_score'][grid_search.best_index_] - grid_search.best_score_
rf_gap_prop = rf_model_gap / abs(grid_search.cv_results_['mean_train_score'][grid_search.best_index_])

In [ ]:
print("> ROOT MEAN SQUARED LOG ERROR: ",round(root_mean_squared_log_error(y_true=y_train, y_pred=y_pred), 4), "\n")
print("> SCORE:", model_best_rf_score, "\n")
print("> BEST HYPERPARAMETERS: ", model_best_rf, "\n")
print("> RF MODEL:", rf_model_gap, "\n")
print("> RF GAP PROP: ", rf_gap_prop*100)